# BD-USBench — Kaggle training notebook

Trains and evaluates all tracks on a Kaggle GPU (T4/P100, 16 GB):

1. Data report
2. **Track A** — supervised classifiers (DINOv2 ViT-S + ConvNeXt-Tiny; linear probe then fine-tune)
3. **Track B** — zero-shot BiomedCLIP and Qwen2.5-VL (4-bit)
4. **Track C** — LoRA fine-tune of Qwen2.5-VL
5. Evaluation per track + comparison table
6. Saves `results/` and the exported model to `/kaggle/working`

**Setup:** attach your private image dataset (images + `metadata.csv`) as a Kaggle dataset,
and attach the BD-USBench repo either as a second Kaggle dataset or set `REPO_GIT_URL`.
Everything is parameterized by the config cell below. Still-image phase only — no live inference.

## Session plan: three Kaggle GPU sessions

Run the tracks in separate sessions so each fits comfortably in a Kaggle GPU session:

1. **Session 1 — Track A** (the default run flags below): the four supervised classifiers.
2. **Session 2 — Track B**: set `RUN_TRACK_A = False`, `RUN_TRACK_B_CLIP = True`, `RUN_TRACK_B_QWEN = True`. Before the Qwen run, install bitsandbytes (see the note in the install cell).
3. **Session 3 — Track C**: set only `RUN_TRACK_C_LORA = True` (also needs bitsandbytes).

Each session re-clones the repo and rebuilds `classes.json` from the same `metadata.csv` with the
same `MIN_TRAIN_IMAGES`, so the class-index mapping is identical across sessions and tracks.

In [ ]:
# ===================== PARAMETERS (edit me) =====================
# Kaggle nests attached datasets under /kaggle/input/datasets/<user>/<slug>.
# The ultrasound dataset is FLAT: all JPGs + metadata.csv sit at DATA_DIR root
# (there is no images/ subfolder).
DATA_DIR     = "/kaggle/input/datasets/kazitahsinraihan/ultrasound-test1"
METADATA_CSV = f"{DATA_DIR}/metadata.csv"
IMAGE_DIR    = DATA_DIR

# Where the repo comes from: a git URL, or a Kaggle dataset containing the repo
REPO_GIT_URL = "https://github.com/ktahsinr/SONAR.git"
REPO_DATASET_DIR = "/kaggle/input/bd-usbench-repo/BD-USBench"  # used if REPO_GIT_URL empty

# Training knobs
EPOCHS_TRACK_A = 30
BATCH_SIZE_TRACK_A = 16
EPOCHS_LORA = 3
DOMAIN = "all"                                 # fetal | adult | all
MIN_TRAIN_IMAGES = 5                           # classes with fewer train images -> "other"
QWEN_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"     # swap for newer Qwen-VL / MedGemma
CLIP_MODEL = "biomedclip"

# Which parts to run — see the session plan above. Session 1 = Track A only.
RUN_TRACK_A = True
RUN_TRACK_B_CLIP = False
RUN_TRACK_B_QWEN = False
RUN_TRACK_C_LORA = False

WORK = "/kaggle/working"
REPO = f"{WORK}/BD-USBench"
print("Parameters set.")

In [ ]:
# ===================== Fail fast on a wrong dataset path =====================
import os
assert os.path.isfile(METADATA_CSV), (
    f"metadata.csv not found at {METADATA_CSV} — is the Kaggle dataset "
    "attached and DATA_DIR correct? Contents of /kaggle/input: "
    + str(os.listdir("/kaggle/input"))
)
jpgs = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith((".jpg", ".jpeg"))]
print(f"OK: {METADATA_CSV} exists | {len(jpgs)} .jpg/.JPG files in {IMAGE_DIR}")
assert jpgs, f"No .jpg/.JPG images found in {IMAGE_DIR}"

In [ ]:
# ===================== Get the repo =====================
import os, shutil, subprocess
if os.path.exists(REPO):
    shutil.rmtree(REPO)
if REPO_GIT_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_GIT_URL, REPO], check=True)
else:
    shutil.copytree(REPO_DATASET_DIR, REPO)
os.chdir(REPO)
print("Repo at:", os.getcwd())

In [ ]:
# ===================== Install requirements =====================
# Kaggle already ships torch/torchvision built against its CUDA — reinstalling
# the pinned versions is slow and can break CUDA. So torch, torchvision and
# bitsandbytes are filtered out of requirements.txt and Kaggle's preinstalled
# builds are kept. bitsandbytes (4-bit NF4 quantization) is ONLY needed for
# Track B-Qwen / Track C: in those sessions, run after this cell:
#     %pip install -q bitsandbytes==0.45.0
import pathlib
SKIP = ("torch==", "torchvision==", "bitsandbytes")
keep = [
    line for line in pathlib.Path("requirements.txt").read_text().splitlines()
    if line.strip() and not line.strip().startswith("#")
    and not line.strip().startswith(SKIP)
]
req = pathlib.Path("/kaggle/working/requirements_kaggle.txt")
req.write_text("\n".join(keep) + "\n")
print("Installing:", ", ".join(line.split(";")[0].split("==")[0].strip() for line in keep))
%pip install -q -r /kaggle/working/requirements_kaggle.txt
%pip install -q -e . --no-deps
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| CUDA:", torch.cuda.is_available())

In [ ]:
# ===================== Patch configs with notebook parameters =====================
import glob, os, yaml, pathlib
CFG_DIR = f"{WORK}/configs"
pathlib.Path(CFG_DIR).mkdir(exist_ok=True)
for src in glob.glob("bd_usbench/configs/*.yaml"):
    name = pathlib.Path(src).name
    if name == "smoke.yaml":
        continue
    cfg = yaml.safe_load(open(src))
    cfg["data"]["metadata_csv"] = METADATA_CSV
    cfg["data"]["image_dir"] = IMAGE_DIR
    cfg["data"]["classes_json"] = f"{WORK}/artifacts/classes.json"
    cfg["data"]["domain"] = DOMAIN
    cfg["data"]["min_train_images"] = MIN_TRAIN_IMAGES
    cfg["output"]["dir"] = f"{WORK}/results"
    if name.startswith("track_a"):
        cfg["train"]["epochs"] = EPOCHS_TRACK_A
        cfg["train"]["batch_size"] = BATCH_SIZE_TRACK_A
    if "qwen" in name:
        cfg["model"]["model_name"] = QWEN_MODEL
        if name.startswith("track_c"):
            cfg["train"]["epochs"] = EPOCHS_LORA
    if "biomedclip" in name:
        cfg["model"]["clip_model"] = CLIP_MODEL
    yaml.safe_dump(cfg, open(f"{CFG_DIR}/{name}", "w"), sort_keys=False)
print("Patched configs:", sorted(os.listdir(CFG_DIR)))

In [ ]:
# ===================== Data report =====================
!python -m bd_usbench.scripts.data_report --metadata {METADATA_CSV} --image-dir {IMAGE_DIR} --min-train-images {MIN_TRAIN_IMAGES} --out {WORK}/results/data_report.txt

In [ ]:
# ===================== Track A: train both backbones, linear probe then fine-tune =====================
if RUN_TRACK_A:
    for cfg_name in ["track_a_dinov2_linear", "track_a_dinov2_finetune",
                     "track_a_convnext_linear", "track_a_convnext_finetune"]:
        print(f"\n===== Training {cfg_name} =====")
        !python -m bd_usbench.scripts.train --config {CFG_DIR}/{cfg_name}.yaml

In [ ]:
# ===================== Track B: zero-shot baselines =====================
if RUN_TRACK_B_CLIP:
    !python -m bd_usbench.scripts.evaluate --track b --config {CFG_DIR}/track_b_biomedclip.yaml --split test --domain {DOMAIN}
if RUN_TRACK_B_QWEN:
    !python -m bd_usbench.scripts.evaluate --track b --config {CFG_DIR}/track_b_qwen.yaml --split test --domain {DOMAIN}

In [ ]:
# ===================== Track C: LoRA fine-tune + evaluate =====================
if RUN_TRACK_C_LORA:
    !python -m bd_usbench.scripts.train_lora --config {CFG_DIR}/track_c_qwen_lora.yaml
    !python -m bd_usbench.scripts.evaluate --track c --config {CFG_DIR}/track_c_qwen_lora.yaml --split test --domain {DOMAIN}

In [ ]:
# ===================== Track A evaluation on the test split =====================
if RUN_TRACK_A:
    for cfg_name in ["track_a_dinov2_linear", "track_a_dinov2_finetune",
                     "track_a_convnext_linear", "track_a_convnext_finetune"]:
        !python -m bd_usbench.scripts.evaluate --track a --config {CFG_DIR}/{cfg_name}.yaml --split test --domain {DOMAIN}

In [ ]:
# ===================== Compare all runs =====================
!python -m bd_usbench.scripts.compare --results-dir {WORK}/results --out {WORK}/results/comparison.md

In [ ]:
# ===================== Results live in /kaggle/working =====================
# results/<run_name>/: checkpoints, export/ (TorchScript + export_meta.json),
# eval_test_*/ (metrics.json, summary.md, confusion_matrix.png, ...),
# results/comparison.md, results/data_report.txt
import subprocess
print(subprocess.run(["find", f"{WORK}/results", "-maxdepth", "3", "-type", "d"],
                     capture_output=True, text=True).stdout)
print(open(f"{WORK}/results/comparison.md").read())